# Phase 11: Production Monitoring & Drift Detection Engine
## MarketPulse: E-Commerce Demand Forecasting Engine

Deploying machine learning models into production is only the beginning of the lifecycle. Once deployed, forecasting models encounter dynamic consumer behavior, macroeconomic shifts, supply shocks, promotional campaigns, and assortment updates.

This notebook demonstrates the **MarketPulse Production Monitoring Engine**, built according to enterprise reliability and model governance standards (Walmart / Amazon / Flipkart):

1. **Covariate Shift Detection**: Population Stability Index (PSI), two-sample Kolmogorov-Smirnov (KS) tests, and Wasserstein (Earth Mover's) distance for price dynamics, discount depth, and calendar features.
2. **Target Demand Dynamics**: Demand sparsity shifts, volume changes, and Syntetos-Boylan demand regime migrations (Smooth, Intermittent, Erratic, Lumpy).
3. **Forecast Accuracy Tracking & Bias Diagnostics**: Tracking Signal ($TS = CFE / MAD$) and two-sided Tabular Cumulative Sum (CUSUM) control charts for early detection of structural residual shifts.
4. **Objective Retraining Decision Matrix**: A 4-tier statistically justified governance framework (`NO_ACTION`, `WARNING`, `RETRAIN_RECOMMENDED`, `CRITICAL_FALLBACK`) that eliminates unnecessary retraining while safeguarding supply chain operations.

In [ ]:
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.price_features import calculate_price_features
from src.features.temporal_features import extract_calendar_features
from src.features.ts_features import classify_demand_pattern
from src.monitoring.drift_detection import (
    calculate_psi,
    classify_psi,
    kolmogorov_smirnov_test,
    wasserstein_distance_test,
    DataDriftDetector,
)
from src.monitoring.performance_monitor import (
    calculate_tracking_signal,
    cusum_drift_detector,
    ModelPerformanceMonitor,
    RetrainingStatus,
)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.titlesize'] = 12

print("MarketPulse Production Monitoring Engine initialized successfully.")

--- 
## 1. Timeline Partitioning: Reference Baseline vs. Operational Windows
We load the M5 processed dataset and augment it with causal expanding price features and calendar indicators. We partition the timeline into:
- **Reference Baseline (Days 1–60)**: Historical in-control training and validation regime.
- **Operational Window 1 (Days 61–75)**: Normal production operations with well-calibrated forecasts.
- **Operational Window 2 (Days 76–90)**: Demand shock / promotional surge where model under-forecasts.

In [ ]:
data_path = PROJECT_ROOT / 'data' / 'processed' / 'm5_processed.parquet'
df = pd.read_parquet(data_path)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['id', 'date']).reset_index(drop=True)

# Feature engineering
df = calculate_price_features(df, date_col='date', use_expanding_max=True)
df = extract_calendar_features(df, date_col='date')

unique_dates = np.sort(df['date'].unique())
ref_dates = unique_dates[:60]
live1_dates = unique_dates[60:75]
live2_dates = unique_dates[75:]

df_ref = df[df['date'].isin(ref_dates)].copy()
df_live1 = df[df['date'].isin(live1_dates)].copy()
df_live2 = df[df['date'].isin(live2_dates)].copy()

print(f"Reference Baseline: {len(df_ref)} records ({pd.to_datetime(ref_dates[0]).date()} to {pd.to_datetime(ref_dates[-1]).date()})")
print(f"Live Window 1:     {len(df_live1)} records ({pd.to_datetime(live1_dates[0]).date()} to {pd.to_datetime(live1_dates[-1]).date()})")
print(f"Live Window 2:     {len(df_live2)} records ({pd.to_datetime(live2_dates[0]).date()} to {pd.to_datetime(live2_dates[-1]).date()})")

--- 
## 2. Covariate Shift & Population Stability Index (PSI)
The Population Stability Index measures changes in the distribution of an input feature between the reference baseline and current production data:

$$PSI = \sum_{b=1}^{B} (P_b - Q_b) \times \ln\left(\frac{P_b}{Q_b}\right)$$

Standard regulatory thresholds:
- $PSI < 0.10$: Stable (insignificant shift)
- $0.10 \le PSI < 0.25$: Moderate drift (warning)
- $PSI \ge 0.25$: Severe covariate drift (action required)

In [ ]:
detector = DataDriftDetector(reference_df=df_ref, alpha=0.05)
monitor_features = ['sell_price', 'discount_depth', 'snap_active', 'is_weekend', 'dayofweek']

drift_w1 = detector.evaluate_feature_drift(df_live1, features=monitor_features)
drift_w1['window'] = 'Window 1 (Days 61-75)'

# Simulate promotional price drop in Window 2 to demonstrate covariate shift
df_live2_promo = df_live2.copy()
df_live2_promo['sell_price'] = df_live2_promo['sell_price'] * 0.75  # 25% price discount
df_live2_promo['discount_depth'] = 0.25

drift_w2 = detector.evaluate_feature_drift(df_live2_promo, features=monitor_features)
drift_w2['window'] = 'Window 2 (Promotional Shift)'

drift_combined = pd.concat([drift_w1, drift_w2], ignore_index=True)
drift_summary = drift_combined[['window', 'feature_name', 'psi', 'psi_status', 'ks_statistic', 'ks_p_value', 'wasserstein_dist']]
drift_summary

In [ ]:
# Visualizing PSI across features and windows
fig, ax = plt.subplots(figsize=(10, 5))
pivoted_psi = drift_combined.pivot(index='feature_name', columns='window', values='psi')
pivoted_psi.plot(kind='barh', ax=ax, colormap='viridis', edgecolor='black', alpha=0.85)

ax.axvline(x=0.10, color='orange', linestyle='--', linewidth=1.5, label='Moderate Drift Threshold (0.10)')
ax.axvline(x=0.25, color='red', linestyle='--', linewidth=1.5, label='Severe Drift Threshold (0.25)')
ax.set_title("Covariate Shift: Population Stability Index (PSI) Across Features", fontsize=13, fontweight='bold')
ax.set_xlabel("PSI Value")
ax.set_ylabel("Feature")
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

--- 
## 3. Target Demand Dynamics & Syntetos-Boylan Regime Migrations
Demand patterns in e-commerce are categorized into four quadrants based on Average Demand Interval ($ADI$) and Squared Coefficient of Variation ($CV^2$):
- **Smooth**: Regular interval ($ADI < 1.32$) and low variance ($CV^2 < 0.49$).
- **Intermittent**: Sporadic demand ($ADI \ge 1.32$) with low variance ($CV^2 < 0.49$).
- **Erratic**: Regular interval ($ADI < 1.32$) but high lumpiness ($CV^2 \ge 0.49$).
- **Lumpy**: Sporadic demand ($ADI \ge 1.32$) and high lumpiness ($CV^2 \ge 0.49$).

In [ ]:
target_shift_w1 = detector.evaluate_target_drift(df_live1, target_col='sales', group_col='id')
target_shift_w2 = detector.evaluate_target_drift(df_live2_promo, target_col='sales', group_col='id')

regime_df = pd.DataFrame([
    {
        'Window': 'Reference Baseline',
        'Zero_Demand_Fraction': target_shift_w1['reference_metrics']['zero_demand_fraction'],
        'Mean_Sales': target_shift_w1['reference_metrics']['mean_sales'],
        'Dominant_Regime': target_shift_w1['reference_metrics']['dominant_regime'],
    },
    {
        'Window': 'Live Window 1',
        'Zero_Demand_Fraction': target_shift_w1['current_metrics']['zero_demand_fraction'],
        'Mean_Sales': target_shift_w1['current_metrics']['mean_sales'],
        'Dominant_Regime': target_shift_w1['current_metrics']['dominant_regime'],
    },
    {
        'Window': 'Live Window 2',
        'Zero_Demand_Fraction': target_shift_w2['current_metrics']['zero_demand_fraction'],
        'Mean_Sales': target_shift_w2['current_metrics']['mean_sales'],
        'Dominant_Regime': target_shift_w2['current_metrics']['dominant_regime'],
    }
])
regime_df

--- 
## 4. Live Model Accuracy Tracking, Tracking Signal ($TS$), and CUSUM
We evaluate the operational predictions against ground truth:
1. **Tracking Signal**: $TS_t = \frac{CFE_t}{MAD_t + \epsilon}$, where $CFE_t = \sum_{\tau=1}^t (y_\tau - \hat{y}_\tau)$.
2. **Tabular CUSUM**: Detects subtle structural mean shifts in forecast errors with slack $k=0.5$ and decision limit $h=4.0$.

In [ ]:
rng = np.random.default_rng(42)

# Window 1: Well-calibrated operational forecast
acts_w1 = df_live1['sales'].values
preds_w1 = np.maximum(0.0, acts_w1 + rng.normal(0, 0.4, size=len(acts_w1)))
preds_w1 = np.maximum(0.0, preds_w1 - np.mean(preds_w1 - acts_w1))

from src.evaluation.metrics import weighted_absolute_percentage_error, mean_absolute_error, root_mean_squared_error
base_metrics = {
    'WAPE': round(float(weighted_absolute_percentage_error(acts_w1, preds_w1)), 2),
    'MAE': round(float(mean_absolute_error(acts_w1, preds_w1)), 3),
    'RMSE': round(float(root_mean_squared_error(acts_w1, preds_w1)), 3),
}

monitor = ModelPerformanceMonitor(baseline_metrics=base_metrics, model_name='Champion_LightGBM')
rec_w1 = monitor.record_evaluation_window(acts_w1, preds_w1, window_id='Window 1', dates=df_live1['date'].values)

# Window 2: Under-forecasting shock (model captures only 75% of promotional surge)
acts_w2 = df_live2['sales'].values
preds_w2 = np.maximum(0.0, acts_w2 * 0.75 + rng.normal(-0.2, 0.6, size=len(acts_w2)))
rec_w2 = monitor.record_evaluation_window(acts_w2, preds_w2, window_id='Window 2', dates=df_live2['date'].values)

pd.DataFrame(monitor.history)

In [ ]:
# Temporal Portfolio Tracking Signal & CUSUM Plots
daily_w1 = pd.DataFrame({'date': df_live1['date'], 'act': acts_w1, 'pred': preds_w1}).groupby('date').sum()
daily_w2 = pd.DataFrame({'date': df_live2['date'], 'act': acts_w2, 'pred': preds_w2}).groupby('date').sum()

cfe1, mad1, ts1 = calculate_tracking_signal(daily_w1['act'].values, daily_w1['pred'].values)
cfe2, mad2, ts2 = calculate_tracking_signal(daily_w2['act'].values, daily_w2['pred'].values)
cusum2 = cusum_drift_detector(daily_w2['act'].values - daily_w2['pred'].values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Tracking Signal Comparison
axes[0].plot(range(1, len(ts1) + 1), ts1, marker='o', color='teal', label='Window 1 (In-Control TS)')
axes[0].plot(range(1, len(ts2) + 1), ts2, marker='s', color='crimson', label='Window 2 (Biased TS)')
axes[0].axhline(y=3.5, color='orange', linestyle='--', label='Warning Limit (+3.5)')
axes[0].axhline(y=-3.5, color='orange', linestyle='--')
axes[0].axhline(y=4.0, color='red', linestyle=':', label='Action Limit (+4.0)')
axes[0].axhline(y=-4.0, color='red', linestyle=':')
axes[0].set_title('Tracking Signal Trajectory Over Time', fontweight='bold')
axes[0].set_xlabel('Day in Window')
axes[0].set_ylabel('Tracking Signal (TS)')
axes[0].legend()

# Plot 2: CUSUM Control Chart for Window 2
axes[1].plot(range(len(cusum2['s_pos'])), cusum2['s_pos'], marker='^', color='darkred', label='S+ (Positive Cumulative Sum)')
axes[1].plot(range(len(cusum2['s_neg'])), cusum2['s_neg'], marker='v', color='navy', label='S- (Negative Cumulative Sum)')
axes[1].axhline(y=4.0, color='red', linestyle='--', label='CUSUM Threshold h=4.0')
if cusum2['change_point_idx'] is not None:
    axes[1].axvline(x=cusum2['change_point_idx'], color='purple', linestyle='-.', label=f"Change Point (Day {cusum2['change_point_idx']})")
axes[1].set_title('Window 2: Tabular CUSUM Control Chart', fontweight='bold')
axes[1].set_xlabel('Day in Window')
axes[1].set_ylabel('CUSUM Statistic')
axes[1].legend()

plt.tight_layout()
plt.show()

--- 
## 5. Statistical Retraining Decision Matrix
Rather than blindly retraining models on fixed cron schedules or ignoring drift, MarketPulse implements an objective 4-tier decision matrix that requires statistical justification before triggering retraining.

In [ ]:
dec_w1 = monitor.evaluate_retraining_necessity(
    current_actuals=acts_w1,
    current_predictions=preds_w1,
    dates=df_live1['date'].values,
    max_feature_psi=float(drift_w1['psi'].max()),
    target_psi=float(target_shift_w1['target_psi']),
)

dec_w2 = monitor.evaluate_retraining_necessity(
    current_actuals=acts_w2,
    current_predictions=preds_w2,
    dates=df_live2['date'].values,
    max_feature_psi=float(drift_w2['psi'].max()),
    target_psi=float(target_shift_w2['target_psi']),
)

print("=" * 80)
print("WINDOW 1 RETRAINING DECISION AUDIT:")
print(f"Decision:                {dec_w1.decision.value}")
print(f"Retraining Justified:    {dec_w1.is_retraining_justified}")
print(f"Urgency:                 {dec_w1.urgency}")
print(f"Delta WAPE:              {dec_w1.delta_wape_pct}%")
print(f"Primary Reason:          {dec_w1.primary_reason}")
print("-" * 80)
print("WINDOW 2 RETRAINING DECISION AUDIT:")
print(f"Decision:                {dec_w2.decision.value}")
print(f"Retraining Justified:    {dec_w2.is_retraining_justified}")
print(f"Urgency:                 {dec_w2.urgency}")
print(f"Delta WAPE:              {dec_w2.delta_wape_pct}%")
print(f"Tracking Signal:         {dec_w2.latest_tracking_signal}")
print(f"Primary Reason:          {dec_w2.primary_reason}")
print(f"Recommended Action:      {dec_w2.recommended_action}")
print("=" * 80)